In [1]:
import pandas as pd

In [125]:
data=pd.read_csv('database_gas.csv')

In [126]:
data['timestamp'] = pd.to_datetime(data['timestamp'])

In [127]:
data['timestamp'] = data['timestamp'].dt.floor(freq='4H')

In [129]:
data.head()

,timestamp,temperature,humidity,CO2CosIRValue,CO2MG811Value,MOX1,MOX2,MOX3,MOX4,COValue
0,2019-11-06 08:00:00+01:00,19.48,54.86,128.0,563.0,476.0,731.0,649.0,565.0,128.0
1,2019-11-06 08:00:00+01:00,19.59,54.23,129.0,563.0,477.0,731.0,649.0,565.0,125.0
2,2019-11-06 08:00:00+01:00,19.63,54.05,128.0,566.0,478.0,732.0,649.0,565.0,125.0
3,2019-11-06 08:00:00+01:00,19.64,53.74,128.0,566.0,478.0,732.0,649.0,565.0,125.0
4,2019-11-06 08:00:00+01:00,19.67,53.53,128.0,569.0,480.0,732.0,650.0,565.0,125.0


In [137]:
df = data.groupby(by='timestamp').count()

In [146]:
df

,temperature,humidity,CO2CosIRValue,CO2MG811Value,MOX1,MOX2,MOX3,MOX4,COValue
timestamp,,,,,,,,,
2019-11-06 08:00:00+01:00,69,69,69,69,69,69,69,69,69
2019-11-06 12:00:00+01:00,720,720,720,720,720,720,720,720,720
2019-11-06 16:00:00+01:00,719,719,719,719,719,719,719,719,719
2019-11-06 20:00:00+01:00,720,720,720,720,720,720,720,720,720
2019-11-07 00:00:00+01:00,720,720,720,720,720,720,720,720,720
...,...,...,...,...,...,...,...,...,...
2020-02-12 16:00:00+01:00,720,720,720,720,720,720,720,720,720
2020-02-12 20:00:00+01:00,720,720,720,720,720,720,720,720,720
2020-02-13 00:00:00+01:00,720,720,720,720,720,720,720,720,720


In [138]:
Unique_row = df.temperature.unique()

In [139]:
Unique_row

array([ 69, 720, 719, 721, 717, 710, 711,   5,  83, 718, 712, 702, 715,
       642, 695, 608, 684, 688, 407, 634, 603])

In [135]:
df

,temperature,humidity,CO2CosIRValue,CO2MG811Value,MOX1,MOX2,MOX3,MOX4,COValue
timestamp,,,,,,,,,
2019-11-06 08:00:00+01:00,69,69,69,69,69,69,69,69,69
2019-11-06 12:00:00+01:00,720,720,720,720,720,720,720,720,720
2019-11-06 16:00:00+01:00,719,719,719,719,719,719,719,719,719
2019-11-06 20:00:00+01:00,720,720,720,720,720,720,720,720,720
2019-11-07 00:00:00+01:00,720,720,720,720,720,720,720,720,720
...,...,...,...,...,...,...,...,...,...
2020-02-12 16:00:00+01:00,720,720,720,720,720,720,720,720,720
2020-02-12 20:00:00+01:00,720,720,720,720,720,720,720,720,720
2020-02-13 00:00:00+01:00,720,720,720,720,720,720,720,720,720


In [128]:
Unique_time = data.timestamp.unique()

In [143]:
len(Unique_time)

582

In [212]:
def convert_to_list(column_name):
    X = []
    column_name=column_name
    for time in Unique_time:
        values = data[data.timestamp==time][column_name].tolist()
        if len(values)<721:
            resample_len=721-len(values)
            new_data = np.random.choice(values, resample_len)
            values.extend(new_data)
        X.append(values)
    return np.array(X)

In [213]:
humidity=convert_to_list(column_name='humidity')

In [214]:
humidity.shape

(582, 721)

In [221]:
def make_feature(column_name):
    column_name=column_name
    x=convert_to_list(column_name=column_name)
    x_stat=stat_area_features(x)
    x_stat=pd.DataFrame(x_stat)
    var = column_name
    column1=[var+'_stat_'+ str(I) for I in range(x_stat.shape[1])]
    x_stat.columns=column1
    x_domain=frequency_domain_features(x)
    x_domain=pd.DataFrame(x_domain)
    column2=[var+'_tm_dom_'+ str(I) for I in range(x_domain.shape[1])]
    x_domain.columns=column2
    x_final=pd.concat([x_stat, x_domain], axis=1)
    return x_final

In [222]:
humidity=make_feature(column_name='humidity')

In [225]:
temperature1=make_feature(column_name='temperature')

In [226]:
temperature1

,temperature_stat_0,temperature_stat_1,temperature_stat_2,temperature_stat_3,temperature_stat_4,temperature_stat_5,temperature_stat_6,temperature_stat_7,temperature_stat_8,temperature_stat_9,...,temperature_tm_dom_16,temperature_tm_dom_17,temperature_tm_dom_18,temperature_tm_dom_19,temperature_tm_dom_20,temperature_tm_dom_21,temperature_tm_dom_22,temperature_tm_dom_23,temperature_tm_dom_24,temperature_tm_dom_25
0,19.788890,20.20,19.48,0.214980,0.372191,-0.996830,0.33,0.16,14247.950,281983.29970,...,13.497809,12.809993,12.755087,12.509823,12.432799,0.087379,0.393897,0.011096,0.015257,0.454924
1,19.936491,20.88,19.00,0.475708,0.055329,-1.002948,0.75,0.39,14354.410,286342.42530,...,59.077658,53.460515,45.366104,37.020960,36.513504,0.004161,0.022191,0.008322,0.018031,0.029126
2,21.993370,24.96,19.11,1.779099,-0.166846,-1.287550,3.64,1.68,15836.305,350596.90315,...,109.647957,86.212853,65.379288,51.221210,39.511469,0.006935,0.012483,0.015257,0.019417,0.023578
3,23.158266,25.12,20.92,1.403190,-0.148739,-1.497803,2.77,1.37,16672.130,387471.72630,...,240.161183,83.724839,69.990292,43.903304,41.491192,0.004161,0.011096,0.019417,0.022191,0.027739
4,20.360624,20.99,19.74,0.322875,0.313755,-0.869116,0.50,0.23,14659.465,298547.06805,...,33.057138,22.357006,20.701138,14.651780,11.661497,0.008322,0.015257,0.011096,0.018031,0.026352
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
577,20.524244,20.81,20.27,0.148500,-0.046014,-1.378833,0.28,0.14,14777.440,303311.32180,...,53.741469,23.445422,19.757331,16.027403,13.565190,0.002774,0.009709,0.012483,0.006935,0.015257
578,20.516061,20.85,20.18,0.157647,-0.020753,-0.908209,0.25,0.12,14771.545,303071.43735,...,40.431944,26.310602,23.385161,21.009162,16.995619,0.002774,0.009709,0.006935,0.012483,0.015257
579,20.487393,20.74,20.16,0.124694,-0.393126,-0.146061,0.18,0.09,14750.995,302222.11225,...,28.377177,21.629593,17.649036,17.256928,16.175432,0.006935,0.009709,0.012483,0.002774,0.015257
580,20.406657,20.69,20.07,0.160806,-0.025523,-1.133705,0.28,0.14,14692.795,299849.49875,...,47.064455,33.067816,28.224740,14.282950,14.223363,0.002774,0.006935,0.009709,0.013870,0.016644


In [223]:
humidity.shape

(582, 36)

In [224]:
humidity

,humidity_stat_0,humidity_stat_1,humidity_stat_2,humidity_stat_3,humidity_stat_4,humidity_stat_5,humidity_stat_6,humidity_stat_7,humidity_stat_8,humidity_stat_9,...,humidity_tm_dom_16,humidity_tm_dom_17,humidity_tm_dom_18,humidity_tm_dom_19,humidity_tm_dom_20,humidity_tm_dom_21,humidity_tm_dom_22,humidity_tm_dom_23,humidity_tm_dom_24,humidity_tm_dom_25
0,51.845409,54.86,47.84,2.511745,-0.413878,-1.632224,4.84,1.15,37328.900,1.939881e+06,...,219.920743,147.461968,143.243092,142.391955,141.307755,0.012483,0.402219,0.384189,0.058252,0.239945
1,50.239667,53.03,46.07,1.352917,-0.179071,-0.280759,1.80,0.89,36173.570,1.818716e+06,...,371.418810,192.462376,149.006799,142.573099,94.932090,0.002774,0.006935,0.009709,0.023578,0.015257
2,51.337295,54.72,46.62,1.827572,-0.246992,-0.706788,2.89,1.41,36962.810,1.899974e+06,...,150.442144,145.309570,97.844703,83.091587,59.834058,0.008322,0.012483,0.015257,0.019417,0.031900
3,51.507725,56.37,46.34,3.170509,0.029583,-1.429598,6.02,3.00,37090.315,1.917907e+06,...,646.070294,402.478886,198.875176,173.025572,137.660801,0.004161,0.006935,0.011096,0.019417,0.022191
4,56.479182,57.17,55.44,0.389339,-0.677460,-0.177516,0.46,0.24,40664.885,2.296822e+06,...,61.529602,60.417079,24.731854,24.243829,23.242392,0.004161,0.008322,0.015257,0.026352,0.012483
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
577,58.626741,59.21,57.32,0.394407,-0.822452,0.307717,0.59,0.28,42212.240,2.474935e+06,...,117.679081,65.288058,64.296666,52.137670,41.209006,0.002774,0.012483,0.009709,0.005548,0.015257
578,58.630277,59.30,57.17,0.428122,-0.967534,0.974247,0.51,0.26,42213.825,2.475142e+06,...,74.005506,57.549199,53.503274,39.387209,38.804282,0.009709,0.012483,0.015257,0.024965,0.018031
579,58.580971,59.14,57.48,0.305984,-1.011760,1.548525,0.32,0.16,42178.630,2.470952e+06,...,65.211887,53.231809,50.060336,46.852292,44.079135,0.006935,0.002774,0.009709,0.015257,0.012483
580,58.430693,59.17,57.51,0.369922,-0.596924,-0.276395,0.48,0.24,42069.950,2.458266e+06,...,115.375307,67.271032,53.562154,45.988048,44.171114,0.002774,0.009709,0.006935,0.016644,0.013870


In [169]:
temperature = np.array(temperature)

In [170]:
temperature.shape

(582, 721)

In [171]:
temp_mean=np.mean(temperature,axis=1)

In [163]:
import scipy.stats as st
from scipy.fftpack import fft, fftfreq 
from scipy.signal import argrelextrema
import operator

In [173]:
def stat_area_features(x, Te=1.0):

    mean_ts = np.mean(x, axis=1).reshape(-1,1) # mean
    max_ts = np.amax(x, axis=1).reshape(-1,1) # max
    min_ts = np.amin(x, axis=1).reshape(-1,1) # min
    std_ts = np.std(x, axis=1).reshape(-1,1) # std
    skew_ts = st.skew(x, axis=1).reshape(-1,1) # skew
    kurtosis_ts = st.kurtosis(x, axis=1).reshape(-1,1) # kurtosis 
    iqr_ts = st.iqr(x, axis=1).reshape(-1,1) # interquartile rante
    mad_ts = np.median(np.sort(abs(x - np.median(x, axis=1).reshape(-1,1)),
                               axis=1), axis=1).reshape(-1,1) # median absolute deviation
    area_ts = np.trapz(x, axis=1, dx=Te).reshape(-1,1) # area under curve
    sq_area_ts = np.trapz(x ** 2, axis=1, dx=Te).reshape(-1,1) # area under curve ** 2

    return np.concatenate((mean_ts,max_ts,min_ts,std_ts,skew_ts,kurtosis_ts,
                           iqr_ts,mad_ts,area_ts,sq_area_ts), axis=1)

In [174]:
temperature_feature=stat_area_features(temperature)

In [177]:
df=pd.DataFrame(temperature_feature)

In [179]:
column=['mean','max','min','std','skew','kurtosis','IR','msd','area_under_curve','sq_area_curve']

In [180]:
df.columns=column

In [181]:
df

,mean,max,min,std,skew,kurtosis,IR,msd,area_under_curve,sq_area_curve
0,19.793689,20.20,19.48,0.213773,0.349661,-1.050770,0.33,0.16,14251.530,282124.70150
1,19.937642,20.88,19.00,0.476235,0.051256,-1.008002,0.77,0.39,14354.825,286359.15395
2,21.996283,24.96,19.11,1.780912,-0.168843,-1.290581,3.64,1.68,15838.095,350680.02075
3,23.154244,25.12,20.92,1.402118,-0.144147,-1.498376,2.77,1.37,16670.680,387403.48930
4,20.361734,20.99,19.74,0.323489,0.310091,-0.875600,0.52,0.23,14659.865,298563.50005
...,...,...,...,...,...,...,...,...,...,...
577,20.524785,20.81,20.27,0.148538,-0.050570,-1.378019,0.28,0.14,14777.635,303319.33045
578,20.516241,20.85,20.18,0.157622,-0.023938,-0.906937,0.25,0.12,14771.610,303074.10170
579,20.487753,20.74,20.16,0.124788,-0.397182,-0.146602,0.18,0.09,14751.125,302227.44745
580,20.406741,20.69,20.07,0.160761,-0.026628,-1.131930,0.28,0.14,14692.825,299850.71795


In [182]:
def frequency_domain_features(x, Te=1.0):

    # As the DFT coefficients and their corresponding frequencies are symetrical arrays
    # with respect to the middle of the array we need to know if the number of readings 
    # in x is even or odd to then split the arrays...
    if x.shape[1]%2 == 0:
        N = int(x.shape[1]/2)
    else:
        N = int(x.shape[1]/2) - 1
    xf = np.repeat(fftfreq(x.shape[1],d=Te)[:N].reshape(1,-1), x.shape[0], axis=0) # frequencies
    dft = np.abs(fft(x, axis=1))[:,:N] # DFT coefficients   
    
    # statistical and area features
    dft_features = stat_area_features(dft, Te=1.0)
    # weighted mean frequency
    dft_weighted_mean_f = np.average(xf, axis=1, weights=dft).reshape(-1,1)
    # 5 first DFT coefficients 
    dft_first_coef = dft[:,:5]    
    # 5 first local maxima of DFT coefficients and their corresponding frequencies
    dft_max_coef = np.zeros((x.shape[0],5))
    dft_max_coef_f = np.zeros((x.shape[0],5))
    for row in range(x.shape[0]):
        # finds all local maximas indexes
        extrema_ind = argrelextrema(dft[row,:], np.greater, axis=0) 
        # makes a list of tuples (DFT_i, f_i) of all the local maxima
        # and keeps the 5 biggest...
        extrema_row = sorted([(dft[row,:][j],xf[row,j]) for j in extrema_ind[0]],
                             key=operator.itemgetter(0), reverse=True)[:5] 
        for i, ext in enumerate(extrema_row):
            dft_max_coef[row,i] = ext[0]
            dft_max_coef_f[row,i] = ext[1]    
    
    return np.concatenate((dft_features,dft_weighted_mean_f,dft_first_coef,
                           dft_max_coef,dft_max_coef_f), axis=1)


In [183]:
temperature_feature1=frequency_domain_features(temperature)

In [185]:
temperature_feature1.shape

(582, 26)

In [187]:
df1=pd.DataFrame(temperature_feature1)

In [188]:
df1

,0,1,2,3,4,5,6,7,8,9,...,16,17,18,19,20,21,22,23,24,25
0,44.765596,14271.25,0.405767,751.898135,18.867652,353.993164,3.924587,1.908562,8931.152716,1.018460e+08,...,17.323449,15.559547,12.119887,12.098744,11.827292,0.015257,0.027739,0.316227,0.049931,0.431345
1,44.622869,14375.04,0.049454,757.480581,18.860968,353.824675,2.039334,0.713373,8831.492365,1.033798e+08,...,58.810205,52.646802,46.170315,36.468758,35.685181,0.004161,0.022191,0.008322,0.018031,0.029126
2,54.639369,15859.32,0.241612,836.611123,18.781394,351.777133,4.643519,1.173157,11685.120446,1.265834e+08,...,109.137089,86.042472,64.788785,51.322506,39.851056,0.006935,0.012483,0.015257,0.019417,0.023578
3,55.518452,16694.21,0.192781,880.144780,18.819652,352.767487,3.526630,1.274722,11583.549539,1.398593e+08,...,239.176570,83.609410,69.006310,45.083856,40.338069,0.004161,0.011096,0.019417,0.022191,0.027739
4,43.450591,14680.81,0.155314,773.653212,18.864758,353.920241,0.929469,0.314130,8258.061172,1.077903e+08,...,32.639927,22.595825,20.946857,14.365189,11.527489,0.008322,0.015257,0.011096,0.018031,0.026352
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
577,42.247476,14798.37,0.024368,779.894873,18.867339,353.985282,0.303174,0.113342,7767.570992,1.095016e+08,...,54.120011,23.112304,20.013512,13.905869,8.169835,0.002774,0.009709,0.012483,0.015257,0.018031
578,42.559173,14792.21,0.024908,779.553625,18.867281,353.983823,0.434436,0.169011,7882.593984,1.094112e+08,...,40.467716,26.321775,23.375566,20.964707,17.106737,0.002774,0.009709,0.006935,0.012483,0.015257
579,42.328259,14771.67,0.005785,778.476242,18.867573,353.991160,0.399923,0.173648,7809.843946,1.091052e+08,...,28.368086,21.853817,17.404356,17.027978,16.318577,0.006935,0.009709,0.012483,0.002774,0.015257
580,42.235689,14713.26,0.037481,775.398781,18.867226,353.982440,0.332309,0.149523,7805.818833,1.082467e+08,...,47.076553,33.118813,28.262448,14.247914,14.184387,0.002774,0.006935,0.009709,0.013870,0.016644


In [194]:
var = 'temperature'
column=[var+'frequency_domain'+ str(I) for I in range(df1.shape[1])]

In [195]:
df1.columns=column

In [203]:
df1

,temperaturefrequency_domain0,temperaturefrequency_domain1,temperaturefrequency_domain2,temperaturefrequency_domain3,temperaturefrequency_domain4,temperaturefrequency_domain5,temperaturefrequency_domain6,temperaturefrequency_domain7,temperaturefrequency_domain8,temperaturefrequency_domain9,...,temperaturefrequency_domain16,temperaturefrequency_domain17,temperaturefrequency_domain18,temperaturefrequency_domain19,temperaturefrequency_domain20,temperaturefrequency_domain21,temperaturefrequency_domain22,temperaturefrequency_domain23,temperaturefrequency_domain24,temperaturefrequency_domain25
0,44.765596,14271.25,0.405767,751.898135,18.867652,353.993164,3.924587,1.908562,8931.152716,1.018460e+08,...,17.323449,15.559547,12.119887,12.098744,11.827292,0.015257,0.027739,0.316227,0.049931,0.431345
1,44.622869,14375.04,0.049454,757.480581,18.860968,353.824675,2.039334,0.713373,8831.492365,1.033798e+08,...,58.810205,52.646802,46.170315,36.468758,35.685181,0.004161,0.022191,0.008322,0.018031,0.029126
2,54.639369,15859.32,0.241612,836.611123,18.781394,351.777133,4.643519,1.173157,11685.120446,1.265834e+08,...,109.137089,86.042472,64.788785,51.322506,39.851056,0.006935,0.012483,0.015257,0.019417,0.023578
3,55.518452,16694.21,0.192781,880.144780,18.819652,352.767487,3.526630,1.274722,11583.549539,1.398593e+08,...,239.176570,83.609410,69.006310,45.083856,40.338069,0.004161,0.011096,0.019417,0.022191,0.027739
4,43.450591,14680.81,0.155314,773.653212,18.864758,353.920241,0.929469,0.314130,8258.061172,1.077903e+08,...,32.639927,22.595825,20.946857,14.365189,11.527489,0.008322,0.015257,0.011096,0.018031,0.026352
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
577,42.247476,14798.37,0.024368,779.894873,18.867339,353.985282,0.303174,0.113342,7767.570992,1.095016e+08,...,54.120011,23.112304,20.013512,13.905869,8.169835,0.002774,0.009709,0.012483,0.015257,0.018031
578,42.559173,14792.21,0.024908,779.553625,18.867281,353.983823,0.434436,0.169011,7882.593984,1.094112e+08,...,40.467716,26.321775,23.375566,20.964707,17.106737,0.002774,0.009709,0.006935,0.012483,0.015257
579,42.328259,14771.67,0.005785,778.476242,18.867573,353.991160,0.399923,0.173648,7809.843946,1.091052e+08,...,28.368086,21.853817,17.404356,17.027978,16.318577,0.006935,0.009709,0.012483,0.002774,0.015257
580,42.235689,14713.26,0.037481,775.398781,18.867226,353.982440,0.332309,0.149523,7805.818833,1.082467e+08,...,47.076553,33.118813,28.262448,14.247914,14.184387,0.002774,0.006935,0.009709,0.013870,0.016644


In [204]:
df_temp=pd.concat([df, df1], axis=1)

In [205]:
df_temp

,mean,max,min,std,skew,kurtosis,IR,msd,area_under_curve,sq_area_curve,...,temperaturefrequency_domain16,temperaturefrequency_domain17,temperaturefrequency_domain18,temperaturefrequency_domain19,temperaturefrequency_domain20,temperaturefrequency_domain21,temperaturefrequency_domain22,temperaturefrequency_domain23,temperaturefrequency_domain24,temperaturefrequency_domain25
0,19.793689,20.20,19.48,0.213773,0.349661,-1.050770,0.33,0.16,14251.530,282124.70150,...,17.323449,15.559547,12.119887,12.098744,11.827292,0.015257,0.027739,0.316227,0.049931,0.431345
1,19.937642,20.88,19.00,0.476235,0.051256,-1.008002,0.77,0.39,14354.825,286359.15395,...,58.810205,52.646802,46.170315,36.468758,35.685181,0.004161,0.022191,0.008322,0.018031,0.029126
2,21.996283,24.96,19.11,1.780912,-0.168843,-1.290581,3.64,1.68,15838.095,350680.02075,...,109.137089,86.042472,64.788785,51.322506,39.851056,0.006935,0.012483,0.015257,0.019417,0.023578
3,23.154244,25.12,20.92,1.402118,-0.144147,-1.498376,2.77,1.37,16670.680,387403.48930,...,239.176570,83.609410,69.006310,45.083856,40.338069,0.004161,0.011096,0.019417,0.022191,0.027739
4,20.361734,20.99,19.74,0.323489,0.310091,-0.875600,0.52,0.23,14659.865,298563.50005,...,32.639927,22.595825,20.946857,14.365189,11.527489,0.008322,0.015257,0.011096,0.018031,0.026352
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
577,20.524785,20.81,20.27,0.148538,-0.050570,-1.378019,0.28,0.14,14777.635,303319.33045,...,54.120011,23.112304,20.013512,13.905869,8.169835,0.002774,0.009709,0.012483,0.015257,0.018031
578,20.516241,20.85,20.18,0.157622,-0.023938,-0.906937,0.25,0.12,14771.610,303074.10170,...,40.467716,26.321775,23.375566,20.964707,17.106737,0.002774,0.009709,0.006935,0.012483,0.015257
579,20.487753,20.74,20.16,0.124788,-0.397182,-0.146602,0.18,0.09,14751.125,302227.44745,...,28.368086,21.853817,17.404356,17.027978,16.318577,0.006935,0.009709,0.012483,0.002774,0.015257
580,20.406741,20.69,20.07,0.160761,-0.026628,-1.131930,0.28,0.14,14692.825,299850.71795,...,47.076553,33.118813,28.262448,14.247914,14.184387,0.002774,0.006935,0.009709,0.013870,0.016644
